# Side-by-Side Eval: Base vs Standard DPO vs Weighted DPO

Tests sycophancy resistance across:
- **Held-out prompts** — never seen during training, varying pressure levels 1-4
- **Val set samples** — in-distribution, one per pressure level and type

Each prompt is paired with a **real video** so the model can ground its answers
in visual evidence. For each prompt, all three models respond. Look for:
- **Base**: does it cave under pressure?
- **Standard DPO**: does it hold ground? Does it sound formulaic?
- **Weighted DPO**: same, but compare tone at high pressure levels

## 1. Mount Drive & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%capture
!pip install transformers==4.45.2 peft==0.12.0 accelerate==0.34.2 qwen-vl-utils datasets -q
print('Done')

## 2. Clone Repo

In [ ]:
import os
REPO_DIR = '/content/sycophancy_VidLMs'
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/rupalirajesh/sycophancy_VidLMs.git {REPO_DIR}
%cd {REPO_DIR}

## 3. Download Videos

Eval needs the same videos as training so each prompt is grounded in real visual content.
If you already downloaded them in the training notebook (and the Colab runtime is the same),
skip this cell.

In [ ]:
HF_TOKEN  = 'hf_YOUR_TOKEN_HERE'
VIDEO_DIR = f'{REPO_DIR}/data/videos'
os.makedirs(VIDEO_DIR, exist_ok=True)

!python download_videos.py \
    --train output/val.jsonl \
    --val   output/val.jsonl \
    --video-dir {VIDEO_DIR} \
    --hf-token {HF_TOKEN}

mp4_count = len(list(__import__('pathlib').Path(VIDEO_DIR).glob('*.mp4')))
print(f'\n{mp4_count} .mp4 files in {VIDEO_DIR}')

## 4. Set Model Paths
Point these at your Drive checkpoints.

In [ ]:
BASE_MODEL = 'Qwen/Qwen2-VL-7B-Instruct'
STD_ADAPTER = '/content/drive/MyDrive/sycophancy_dpo/standard/final'
WTD_ADAPTER = '/content/drive/MyDrive/sycophancy_dpo/weighted/final'
VAL_PATH    = f'{REPO_DIR}/output/val.jsonl'
VIDEO_DIR   = f'{REPO_DIR}/data/videos'
OUT_PATH    = '/content/drive/MyDrive/sycophancy_dpo/eval_results.json'

for name, path in [('Standard', STD_ADAPTER), ('Weighted', WTD_ADAPTER)]:
    exists = os.path.isdir(path)
    print(f'{name} adapter: {path}  →  {"✓" if exists else "✗ NOT FOUND"}')

mp4_count = len(list(__import__('pathlib').Path(VIDEO_DIR).glob('*.mp4')))
print(f'Videos: {mp4_count} .mp4 files in {VIDEO_DIR}')

## 5. Run Eval

In [ ]:
!python eval_dpo.py \
    --base      {BASE_MODEL} \
    --std       {STD_ADAPTER} \
    --wtd       {WTD_ADAPTER} \
    --val       {VAL_PATH} \
    --video-dir {VIDEO_DIR} \
    --n-val 6 \
    --out       {OUT_PATH}

## 6. Pretty-Print Results from JSON
Run this cell after eval finishes to re-display any result cleanly.

In [ ]:
import json
from IPython.display import display, HTML

with open(OUT_PATH) as f:
    results = json.load(f)

COLORS = {'Base': '#f0f0f0', 'Standard DPO': '#ddeeff', 'Weighted DPO': '#ddfde8'}

for r in results:
    # Extract last user turn text (handles multimodal content blocks)
    last_msg = next(m for m in reversed(r['messages']) if m['role'] == 'user')
    content = last_msg['content']
    if isinstance(content, list):
        last_user = next((b['text'] for b in content if b.get('type') == 'text'), '')
    else:
        last_user = content

    html = f'<div style="border:2px solid #ccc;border-radius:8px;padding:12px;margin:16px 0">'
    html += f'<b style="font-size:1.05em">{r["label"]}</b><br><br>'
    html += f'<div style="background:#fffbe6;padding:8px;border-radius:4px;margin-bottom:10px">'
    html += f'<b>User:</b> {last_user}</div>'
    for model_name, response in r['responses'].items():
        bg = COLORS.get(model_name, '#ffffff')
        html += f'<div style="background:{bg};padding:8px;border-radius:4px;margin-bottom:6px">'
        html += f'<b>{model_name}:</b> {response}</div>'
    if r.get('ground_truth'):
        html += f'<div style="background:#fafafa;padding:8px;border-radius:4px;color:#555">'
        html += f'<b>Ground truth:</b> {r["ground_truth"]}</div>'
    if r.get('chosen_reference'):
        html += f'<div style="background:#fafafa;padding:8px;border-radius:4px;font-style:italic;color:#555">'
        html += f'<b>Reference (chosen):</b> {r["chosen_reference"]}</div>'
    html += '</div>'
    display(HTML(html))